# 67 — CRM Enrichment Agent

## Goal
Build an agent that **enriches contacts** with information,
**summarizes account context**, and **drafts next actions** — like
an AI-powered CRM assistant.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Multi-step workflow** | Enrich → Summarize → Plan |
| **Data enrichment** | Augment contacts with external info |
| **Account summarization** | Distill interaction history |
| **Action planning** | Suggest next steps based on context |

## Architecture
```
Contact Name/ID → Lookup CRM → Enrich (web search) → Summarize → Draft Actions
```

## Stack
- `LangGraph` + `ChatOllama`
- Simulated CRM database
- DuckDuckGo for enrichment

In [ ]:
import os, warnings, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from datetime import datetime
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from duckduckgo_search import DDGS

print("All imports OK")

## Step 1 — Simulated CRM Database

In [ ]:
CRM_DB = {
    "C001": {
        "name": "Sarah Mitchell",
        "company": "TechNova Inc.",
        "title": "VP of Engineering",
        "email": "sarah.m@technova.example.com",
        "phone": "+1-555-0101",
        "deal_stage": "negotiation",
        "deal_value": 150000,
        "last_contact": "2025-05-28",
        "notes": [
            {"date": "2025-05-28", "text": "Demo went well. Interested in enterprise plan. Needs pricing by Friday."},
            {"date": "2025-05-15", "text": "Intro call. Pain point: current tool doesn't scale beyond 50 users."},
            {"date": "2025-05-01", "text": "Inbound lead from webinar. Downloaded whitepaper on scalability."},
        ],
    },
    "C002": {
        "name": "James Parker",
        "company": "DataFlow Systems",
        "title": "CTO",
        "email": "j.parker@dataflow.example.com",
        "phone": "+1-555-0202",
        "deal_stage": "discovery",
        "deal_value": 75000,
        "last_contact": "2025-06-01",
        "notes": [
            {"date": "2025-06-01", "text": "Follow-up email sent. No response yet."},
            {"date": "2025-05-20", "text": "Cold outreach. Showed interest in API integration features."},
        ],
    },
    "C003": {
        "name": "Lisa Chen",
        "company": "GreenLeaf Analytics",
        "title": "Director of Data Science",
        "email": "lchen@greenleaf.example.com",
        "phone": "+1-555-0303",
        "deal_stage": "closed-won",
        "deal_value": 200000,
        "last_contact": "2025-04-15",
        "notes": [
            {"date": "2025-04-15", "text": "Contract signed! 2-year deal. Onboarding scheduled for May 1."},
            {"date": "2025-04-01", "text": "Final negotiation. Added dedicated support to package."},
            {"date": "2025-03-15", "text": "Technical evaluation complete. Passed all requirements."},
            {"date": "2025-02-28", "text": "Initial meeting. Looking to replace legacy analytics platform."},
        ],
    },
}

_action_plans = {}

print(f"CRM database: {len(CRM_DB)} contacts")
for cid, contact in CRM_DB.items():
    print(f"  {cid}: {contact['name']} ({contact['company']}) — {contact['deal_stage']}")

## Step 2 — Define CRM Tools

In [ ]:
@tool
def lookup_contact(contact_id: str) -> str:
    """Look up a contact by their CRM ID (e.g., C001). Returns full contact info and interaction history."""
    if contact_id not in CRM_DB:
        return f"Contact {contact_id} not found. Available: {list(CRM_DB.keys())}"
    contact = CRM_DB[contact_id]
    return json.dumps(contact, indent=2)

@tool
def search_contacts(query: str) -> str:
    """Search contacts by name, company, or deal stage."""
    results = []
    for cid, contact in CRM_DB.items():
        searchable = f"{contact['name']} {contact['company']} {contact['deal_stage']}".lower()
        if query.lower() in searchable:
            results.append(f"{cid}: {contact['name']} @ {contact['company']} — {contact['deal_stage']} (${contact['deal_value']:,})")
    return "\n".join(results) if results else f"No contacts matching '{query}'."

@tool
def enrich_contact(company_name: str) -> str:
    """Search the web for information about a company to enrich CRM data."""
    try:
        with DDGS() as ddgs:
            results = list(ddgs.text(f"{company_name} company", max_results=3))
        if not results:
            return f"No web results found for {company_name}."
        formatted = [f"[{i+1}] {r['title']}\n    {r['body'][:200]}" for i, r in enumerate(results)]
        return f"Web info about {company_name}:\n" + "\n".join(formatted)
    except Exception as e:
        return f"Search error: {e}"

@tool
def add_note(contact_id: str, note_text: str) -> str:
    """Add a new interaction note to a contact's CRM record."""
    if contact_id not in CRM_DB:
        return f"Contact {contact_id} not found."
    CRM_DB[contact_id]["notes"].insert(0, {
        "date": datetime.now().strftime("%Y-%m-%d"),
        "text": note_text,
    })
    return f"✅ Note added to {contact_id}"

@tool
def save_action_plan(contact_id: str, plan_json: str) -> str:
    """Save a suggested action plan for a contact. plan_json should have keys: priority, next_action, timeline, talking_points."""
    try:
        plan = json.loads(plan_json)
        _action_plans[contact_id] = plan
        return f"✅ Action plan saved for {contact_id}"
    except json.JSONDecodeError as e:
        return f"Invalid JSON: {e}"

crm_tools = [lookup_contact, search_contacts, enrich_contact, add_note, save_action_plan]
print(f"Defined {len(crm_tools)} CRM tools")

## Step 3 — Create the CRM Agent

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

CRM_PROMPT = """You are a CRM enrichment assistant for a B2B sales team.

You help account executives by:
1. Looking up contact and account information
2. Enriching contacts with web research about their company
3. Summarizing the interaction history and account context
4. Suggesting next actions with priority and timeline

Rules:
- Always provide a concise account summary before suggesting actions
- Consider the deal stage when prioritizing actions
- Include specific talking points for each next action
- Note any risks or time-sensitive items
- Save action plans using save_action_plan()"""

crm_agent = create_react_agent(
    model=llm,
    tools=crm_tools,
    prompt=CRM_PROMPT,
)

def ask_crm_agent(request: str):
    print(f"\n{'='*70}")
    print(f"REQUEST: {request}")
    print(f"{'='*70}")
    result = crm_agent.invoke({"messages": [{"role": "user", "content": request}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f"\n🔧 TOOL: {tc['name']}({str(tc['args'])[:120]})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:400]}")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 RESPONSE:\n{msg.content}")
    return result

print("CRM agent ready!")

In [ ]:
# Task 1: Enrich and summarize a contact in negotiation
ask_crm_agent("Give me a full briefing on contact C001 (Sarah Mitchell). Enrich with company info and suggest next actions.")

In [ ]:
# Task 2: Stalled deal follow-up
ask_crm_agent("Contact C002 (James Parker) hasn't responded. Research his company and suggest a re-engagement strategy.")

In [ ]:
# Task 3: Upsell opportunity on closed deal
ask_crm_agent("Contact C003 is a closed-won deal. Look at the history and suggest upsell/cross-sell opportunities.")

In [ ]:
# Review all action plans
print("\n" + "="*70)
print("ACTION PLANS")
print("="*70)
for cid, plan in _action_plans.items():
    print(f"\n{cid} ({CRM_DB[cid]['name']}):")
    print(json.dumps(plan, indent=2))

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **CRM lookup** | Fetch contact info and interaction history |
| **Web enrichment** | DuckDuckGo search for company context |
| **Account summary** | Agent synthesizes notes into narrative |
| **Action planning** | Prioritized next steps with talking points |

## Multi-Agent Extension (CrewAI Pattern)
```
In CrewAI, you could split this into specialized agents:

Agent 1: Researcher    → Web enrichment, company intel
Agent 2: Analyst       → Summarize history, identify patterns
Agent 3: Strategist    → Draft action plan, talking points

Each agent focuses on its specialty, and results are composed.
```